# HRV Stress Detection — Data Mining & ML Coursework (F21DL)

**Dataset:** HRV stress dataset — RRI beat-level recordings + minute-level labeled sessions
(conditions: rest, no stress, time pressure, interruption) across 25 participants.

**Pipeline:**
1. Data cleaning & merging
2. Feature engineering
3. Exploratory Data Analysis
4. Generative modeling (Naive Bayes, GMM/clustering)
5. Discriminative modeling (Decision Tree, Logistic Regression, etc.)
6. Evaluation & comparison


## 1. Data Cleaning & Merging

In [1]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

RAW_DIR = Path("hrv dataset/data/raw")
LABELS_FILE = RAW_DIR / "labels" / "hrv stress labels.xlsx"
RRI_DIR = RAW_DIR / "rri"
PROCESSED_DIR = Path("hrv dataset/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Labels: one sheet per participant (p1..p25), stacked into one table
labels_sheets = pd.read_excel(LABELS_FILE, sheet_name=None)
labels_df = pd.concat(labels_sheets.values(), ignore_index=True)
print("Raw labels shape:", labels_df.shape)
labels_df.head()

Raw labels shape: (3140, 12)


,Unnamed: 0,PP,C,timestamp,HR,RMSSD,SCL,date,subject,label,Condition,ElapsedTime
0,0,PP1,1,20120918T131600000,NaN,NaN,80.239727,2012-09-18 13:16:00,p1,rest,R,0
1,1,PP1,1,20120918T131700000,61.0,0.061420,77.365127,2012-09-18 13:17:00,p1,rest,R,1
2,2,PP1,1,20120918T131800000,64.0,0.049663,77.359559,2012-09-18 13:18:00,p1,rest,R,2
3,3,PP1,1,20120918T131900000,60.0,0.052487,76.728772,2012-09-18 13:19:00,p1,rest,R,3
4,4,PP1,1,20120918T132000000,61.0,0.051189,76.512877,2012-09-18 13:20:00,p1,rest,R,4


### Known data quality issues
- `timestamp` string column is malformed for 7 rows (truncated, e.g. `20121001T162`) — the parsed
  `date` column (datetime64) is reliable for all rows, so we drop `timestamp` and use `date`.
- `HR` and `RMSSD` are missing for 1,650/3,140 rows (~52%). This is **not random**: subjects
  `p8` and `p11` have **100%** missing HR/RMSSD because they have no corresponding RRI file
  (HR/RMSSD are derived from beat-to-beat RR intervals — no ECG recording, no derived values).
- `SCL` (skin conductance level) varies by orders of magnitude between subjects (e.g. p1 ~80 vs
  p9 ~1400) — this is a per-subject baseline/sensor-placement effect, not signal; we normalise
  per-subject with a z-score.


In [2]:
clean = labels_df.drop(columns=["Unnamed: 0", "timestamp"]).copy()
clean = clean.rename(columns={"PP": "pp_code", "C": "session"})

print("Missing values per column:")
print(clean.isna().sum())
clean.to_csv(PROCESSED_DIR / "labels_clean.csv", index=False)

Missing values per column:
pp_code           0
session           0
HR             1650
RMSSD          1650
SCL             562
date              0
subject           0
label             0
Condition         0
ElapsedTime       0
dtype: int64


In [3]:
# Impute HR/RMSSD/SCL: per-subject median first, global median fallback
# (fallback only matters for p8/p11, which have zero RRI-derived signal)
global_med = {c: clean[c].median() for c in ["HR", "RMSSD", "SCL"]}

for col in ["HR", "RMSSD", "SCL"]:
    subj_med = clean.groupby("subject")[col].transform("median")
    clean[f"{col}_imputed_flag"] = clean[col].isna().astype(int)
    clean[f"{col}_filled"] = clean[col].fillna(subj_med).fillna(global_med[col])

# Per-subject z-score for SCL (fixes wildly different baselines across subjects)
clean["SCL_z"] = clean.groupby("subject")["SCL_filled"].transform(lambda s: (s - s.mean()) / s.std())

final_cols = ["subject", "pp_code", "session", "date", "ElapsedTime", "Condition", "label",
              "HR_filled", "RMSSD_filled", "SCL_filled", "SCL_z",
              "HR_imputed_flag", "RMSSD_imputed_flag", "SCL_imputed_flag"]
features_minute = clean[final_cols].rename(
    columns={"HR_filled": "HR", "RMSSD_filled": "RMSSD", "SCL_filled": "SCL"}
)

print("Final shape:", features_minute.shape)
print(features_minute["label"].value_counts())
features_minute.to_csv(PROCESSED_DIR / "features_minute_level.csv", index=False)
features_minute.head()

Final shape: (3140, 14)
label
no stress        1029
interruption      996
time pressure     664
rest              451
Name: count, dtype: int64


,subject,pp_code,session,date,ElapsedTime,Condition,label,HR,RMSSD,SCL,SCL_z,HR_imputed_flag,RMSSD_imputed_flag,SCL_imputed_flag
0,p1,PP1,1,2012-09-18 13:16:00,0,R,rest,65.0,0.048305,80.239727,3.337706,1,1,0
1,p1,PP1,1,2012-09-18 13:17:00,1,R,rest,61.0,0.061420,77.365127,2.680570,0,0,0
2,p1,PP1,1,2012-09-18 13:18:00,2,R,rest,64.0,0.049663,77.359559,2.679298,0,0,0
3,p1,PP1,1,2012-09-18 13:19:00,3,R,rest,60.0,0.052487,76.728772,2.535099,0,0,0
4,p1,PP1,1,2012-09-18 13:20:00,4,R,rest,61.0,0.051189,76.512877,2.485746,0,0,0


### RRI (beat-level) files
25 participants expected; `p8` and `p11` have no RRI file (confirmed above — matches 100% missing
HR/RMSSD for those subjects). Raw RRI time (`time_s`) is seconds since each participant's
recording start and does **not** line up cleanly with the labels' `ElapsedTime` once you account
for breaks between sessions (recording pauses during breaks, elapsed-time clock does not) — so we
use the RRI files for supplementary signal visualisation (tachogram, Poincare plot) rather than
row-level merging into the modeling table.

In [4]:
rri_files = sorted(RRI_DIR.glob("p*.txt"), key=lambda p: int(p.stem[1:]))
print(f"{len(rri_files)} RRI files found:", [f.stem for f in rri_files])

rri_frames = []
for path in rri_files:
    frame = pd.read_csv(path, sep=r"\s+", header=None, names=["time_s", "rri_ms"])
    frame.insert(0, "subject", path.stem)
    rri_frames.append(frame)
rri_df = pd.concat(rri_frames, ignore_index=True)
print("RRI combined shape:", rri_df.shape)
rri_df.to_csv(PROCESSED_DIR / "rri_combined.csv", index=False)
rri_df.head()

23 RRI files found: ['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p9', 'p10', 'p12', 'p13', 'p14', 'p15', 'p16', 'p17', 'p18', 'p19', 'p20', 'p21', 'p22', 'p23', 'p24', 'p25']
RRI combined shape: (704454, 3)


,subject,time_s,rri_ms
0,p1,1.265625,870.11719
1,p1,1.515625,885.36996
2,p1,1.765625,890.18974
3,p1,2.015625,886.73851
4,p1,2.265625,877.17820


## 2. Feature Engineering

### Target definitions
- **Multi-class target**: `label` (4 classes: rest, no stress, time pressure, interruption)
- **Binary target**: `is_stress` — stressors (time pressure, interruption) = 1, baseline
  (rest, no stress) = 0. Based on the SWELL-style experiment design where time pressure and
  interruption are the stress-inducing conditions.

### Derived features (minute-level table)
- `SCL_log` = log1p(SCL) — SCL is heavy-right-tailed (max 1455 vs median 172), log-transform
  stabilises variance for linear/generative models.
- `RMSSD_HR_ratio` — rough parasympathetic/sympathetic balance proxy.
- `label_code` — integer encoding of the 4-class label for models needing numeric targets.


In [5]:
features_minute = pd.read_csv(PROCESSED_DIR / "features_minute_level.csv")

stress_labels = {"time pressure", "interruption"}
features_minute["is_stress"] = features_minute["label"].isin(stress_labels).astype(int)
features_minute["SCL_log"] = np.log1p(features_minute["SCL"])
features_minute["RMSSD_HR_ratio"] = features_minute["RMSSD"] / features_minute["HR"]
features_minute["label_code"] = features_minute["label"].astype("category").cat.codes

features_minute.to_csv(PROCESSED_DIR / "features_engineered.csv", index=False)
print("Engineered shape:", features_minute.shape)
print("\nBinary class balance (is_stress):")
print(features_minute["is_stress"].value_counts())
print("\nMulti-class balance (label):")
print(features_minute["label"].value_counts())
features_minute.head()

Engineered shape: (3140, 18)

Binary class balance (is_stress):
is_stress
1    1660
0    1480
Name: count, dtype: int64

Multi-class balance (label):
label
no stress        1029
interruption      996
time pressure     664
rest              451
Name: count, dtype: int64


,subject,pp_code,session,date,ElapsedTime,Condition,label,HR,RMSSD,SCL,SCL_z,HR_imputed_flag,RMSSD_imputed_flag,SCL_imputed_flag,is_stress,SCL_log,RMSSD_HR_ratio,label_code
0,p1,PP1,1,2012-09-18 13:16:00,0,R,rest,65.0,0.048305,80.239727,3.337706,1,1,0,0,4.397404,0.000743,2
1,p1,PP1,1,2012-09-18 13:17:00,1,R,rest,61.0,0.061420,77.365127,2.680570,0,0,0,0,4.361379,0.001007,2
2,p1,PP1,1,2012-09-18 13:18:00,2,R,rest,64.0,0.049663,77.359559,2.679298,0,0,0,0,4.361308,0.000776,2
3,p1,PP1,1,2012-09-18 13:19:00,3,R,rest,60.0,0.052487,76.728772,2.535099,0,0,0,0,4.353225,0.000875,2
4,p1,PP1,1,2012-09-18 13:20:00,4,R,rest,61.0,0.051189,76.512877,2.485746,0,0,0,0,4.350444,0.000839,2


### RRI-derived subject HRV profiles (supplementary)
Per the Step-1 alignment limitation, we don't merge beat-level RRI into the per-minute table.
Instead we compute one aggregate HRV profile per subject (whole recording) — useful for EDA,
tachogram/Poincare plots, and an extra unsupervised angle (e.g. clustering subjects by HRV
profile) that doesn't depend on fragile time alignment.

In [6]:
def pnn50(x):
    diffs = np.abs(np.diff(x))
    return 100.0 * np.mean(diffs > 50) if len(diffs) else np.nan

rows = []
for subj, g in rri_df.groupby("subject"):
    vals = g["rri_ms"].values
    rows.append({
        "subject": subj,
        "n_beats": len(vals),
        "mean_rri_ms": np.mean(vals),
        "sdnn_ms": np.std(vals),
        "rmssd_ms": np.sqrt(np.mean(np.diff(vals) ** 2)),
        "pnn50_pct": pnn50(vals),
        "mean_hr_bpm": 60000.0 / np.mean(vals),
        "recording_duration_min": (g["time_s"].max() - g["time_s"].min()) / 60.0,
    })
rri_profile = pd.DataFrame(rows).sort_values(
    "subject", key=lambda s: s.str.extract(r"(\d+)")[0].astype(int)
).reset_index(drop=True)
rri_profile.to_csv(PROCESSED_DIR / "rri_subject_profile.csv", index=False)
print(rri_profile.shape)
rri_profile

(23, 8)


,subject,n_beats,mean_rri_ms,sdnn_ms,rmssd_ms,pnn50_pct,mean_hr_bpm,recording_duration_min
0,p1,35974,952.795885,132.584486,12.203245,0.144553,62.972564,149.887500
1,p2,33789,804.744184,69.994513,12.877839,0.112466,74.557855,140.783333
2,p3,30890,721.704345,211.228932,13.613857,0.848198,83.136537,128.704166
3,p4,26655,1040.232213,135.913557,21.228042,2.138516,57.679429,111.058333
4,p5,31006,846.949363,141.565738,16.117432,1.093372,70.842488,129.187500
5,p6,33604,961.377641,142.112834,18.648291,1.276672,62.410438,140.012499
6,p7,29805,848.309676,102.999173,13.211250,0.208026,70.728888,124.183334
7,p9,30578,874.550759,115.272432,15.489993,1.066161,68.606652,127.404167
8,p10,30575,904.929356,205.235095,18.349889,1.625564,66.303518,127.391667
9,p12,34395,771.680560,120.644079,11.490202,0.375065,77.752380,143.308333


## 3. Exploratory Data Analysis

In [7]:
import seaborn as sns
sns.set_theme(style="whitegrid")
FIG_DIR = PROCESSED_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)
label_order = ["rest", "no stress", "time pressure", "interruption"]

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
features_minute["label"].value_counts().loc[label_order].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Class balance (4-class label)"); axes[0].tick_params(axis="x", rotation=30)
features_minute["is_stress"].map({0: "non-stress", 1: "stress"}).value_counts().plot(
    kind="bar", ax=axes[1], color="#DD8452")
axes[1].set_title("Class balance (binary is_stress)")
plt.tight_layout(); plt.savefig(FIG_DIR / "01_class_balance.png", dpi=150); plt.show()

/var/folders/ny/s5wjrp7d0l93sy7y1qnr6sz80000gn/T/ipykernel_9349/655222853.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG_DIR / "01_class_balance.png", dpi=150); plt.show()


Both targets are usable without heavy resampling: the binary target is nearly balanced
(1660 stress / 1480 non-stress); the 4-class target is moderately imbalanced (rest is the
smallest class at 451 rows) — we'll use stratified splits and may report class-weighted
metrics (F1-macro) alongside accuracy.

In [9]:
feats_to_plot = ["HR", "RMSSD", "SCL_log", "RMSSD_HR_ratio"]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.flat, feats_to_plot):
    sns.boxplot(data=features_minute, x="label", y=col, order=label_order, hue="label",
                legend=False, ax=ax, palette="Set2")
    ax.set_title(f"{col} by condition"); ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig(FIG_DIR / "02_feature_distributions.png", dpi=150); plt.show()

/var/folders/ny/s5wjrp7d0l93sy7y1qnr6sz80000gn/T/ipykernel_9349/1517651557.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG_DIR / "02_feature_distributions.png", dpi=150); plt.show()


**Key EDA finding:** class medians for HR/RMSSD/SCL_log/RMSSD_HR_ratio overlap heavily across
all four conditions — minute-level autonomic features alone do **not** separate stress
conditions with a simple linear threshold. This matches the weak linear correlations below and
motivates trying non-linear/tree-based discriminative models rather than relying on linear
decision boundaries.

In [10]:
corr_cols = ["HR", "RMSSD", "SCL", "SCL_log", "SCL_z", "RMSSD_HR_ratio", "label_code", "is_stress"]
corr = features_minute[corr_cols].corr()
plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation matrix")
plt.tight_layout(); plt.savefig(FIG_DIR / "03_correlation_heatmap.png", dpi=150); plt.show()
corr["is_stress"].sort_values(ascending=False)

/var/folders/ny/s5wjrp7d0l93sy7y1qnr6sz80000gn/T/ipykernel_9349/1199499172.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG_DIR / "03_correlation_heatmap.png", dpi=150); plt.show()


is_stress         1.000000
SCL_z             0.085959
RMSSD_HR_ratio    0.067898
RMSSD             0.062441
SCL_log           0.056687
SCL               0.032353
label_code       -0.046863
HR               -0.047664
Name: is_stress, dtype: float64

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for subj in features_minute["subject"].unique()[:8]:
    sns.kdeplot(features_minute.loc[features_minute.subject == subj, "SCL"], ax=axes[0], legend=False)
axes[0].set_title("SCL raw — per-subject distributions (8 subjects)")
for subj in features_minute["subject"].unique()[:8]:
    sns.kdeplot(features_minute.loc[features_minute.subject == subj, "SCL_z"], ax=axes[1], legend=False)
axes[1].set_title("SCL_z — after per-subject normalisation")
plt.tight_layout(); plt.savefig(FIG_DIR / "04_scl_normalization_effect.png", dpi=150); plt.show()

/var/folders/ny/s5wjrp7d0l93sy7y1qnr6sz80000gn/T/ipykernel_9349/1810930403.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG_DIR / "04_scl_normalization_effect.png", dpi=150); plt.show()


Raw SCL distributions are scattered across very different ranges per subject (sensor
baseline/skin conductance differs physiologically between people). After per-subject
z-scoring, all subjects' distributions are centred and comparable — justifying the
normalisation decision made in Step 1.

In [12]:
p1 = rri_df[rri_df.subject == "p1"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(p1["time_s"] / 60, p1["rri_ms"], lw=0.4)
axes[0].set_xlabel("time (min)"); axes[0].set_ylabel("RRI (ms)")
axes[0].set_title("Tachogram — subject p1 (full recording)")
rr = p1["rri_ms"].values
axes[1].scatter(rr[:-1], rr[1:], s=1, alpha=0.3)
axes[1].set_xlabel("RRI(n) ms"); axes[1].set_ylabel("RRI(n+1) ms")
axes[1].set_title("Poincare plot — subject p1")
plt.tight_layout(); plt.savefig(FIG_DIR / "05_rri_signal_examples.png", dpi=150); plt.show()

/var/folders/ny/s5wjrp7d0l93sy7y1qnr6sz80000gn/T/ipykernel_9349/3914398661.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG_DIR / "05_rri_signal_examples.png", dpi=150); plt.show()


In [13]:
plt.figure(figsize=(6, 5))
plt.scatter(rri_profile["mean_hr_bpm"], rri_profile["sdnn_ms"],
            s=rri_profile["recording_duration_min"], alpha=0.6)
for _, r in rri_profile.iterrows():
    plt.annotate(r["subject"], (r["mean_hr_bpm"], r["sdnn_ms"]), fontsize=7)
plt.xlabel("Mean HR (bpm)"); plt.ylabel("SDNN (ms)")
plt.title("Subject-level HRV profile (bubble size = recording duration)")
plt.tight_layout(); plt.savefig(FIG_DIR / "06_subject_hrv_profiles.png", dpi=150); plt.show()

/var/folders/ny/s5wjrp7d0l93sy7y1qnr6sz80000gn/T/ipykernel_9349/251940433.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG_DIR / "06_subject_hrv_profiles.png", dpi=150); plt.show()


## 4. Generative modeling

Generative models learn **p(x | y)** (class-conditional densities) and invert them with Bayes'
rule. We use:

- **Gaussian Naive Bayes** — assumes features are independent Gaussians given the class.
- **Gaussian Mixture Model / k-means** — unsupervised structure discovery (do clusters recover
  the known stress labels?).
- **Class-conditional GMM** — a slightly richer generative classifier (mixture per class
  instead of a single Gaussian).

### Evaluation protocol (used for all later models too)
Minute-level rows from the *same person* are highly dependent. A random row-wise split would
leak that person's physiology into both train and test and inflate accuracy. We therefore use
a **subject-aware split**: `GroupShuffleSplit` / `GroupKFold` grouped by `subject`, so whole
people are held out. Features are standardised on the training fold only.

Feature set: `HR`, `RMSSD`, `SCL_z`, `RMSSD_HR_ratio`.

In [14]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    adjusted_rand_score, normalized_mutual_info_score,
)

feature_cols = ["HR", "RMSSD", "SCL_z", "RMSSD_HR_ratio"]
X = features_minute[feature_cols].to_numpy()
y_bin = features_minute["is_stress"].to_numpy()
y_multi = features_minute["label"].to_numpy()
groups = features_minute["subject"].to_numpy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y_bin, groups))
print("Train subjects:", sorted(set(groups[train_idx])))
print("Held-out test subjects:", sorted(set(groups[test_idx])))
print(f"Train rows: {len(train_idx)}  Test rows: {len(test_idx)}")

scaler = StandardScaler()
X_train = scaler.fit_transform(X[train_idx])
X_test = scaler.transform(X[test_idx])
y_bin_tr, y_bin_te = y_bin[train_idx], y_bin[test_idx]
y_multi_tr, y_multi_te = y_multi[train_idx], y_multi[test_idx]

Train subjects: ['p10', 'p11', 'p12', 'p13', 'p14', 'p15', 'p16', 'p18', 'p19', 'p20', 'p21', 'p22', 'p23', 'p25', 'p3', 'p4', 'p5', 'p6', 'p7', 'p9']
Held-out test subjects: ['p1', 'p17', 'p2', 'p24', 'p8']
Train rows: 2515  Test rows: 625


In [15]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_bin_tr)
dummy_pred = dummy.predict(X_test)
print("Dummy (most-frequent) binary accuracy:", accuracy_score(y_bin_te, dummy_pred))

gnb_bin = GaussianNB()
gnb_bin.fit(X_train, y_bin_tr)
gnb_bin_pred = gnb_bin.predict(X_test)
print("\nGaussian NB — binary")
print("accuracy:", accuracy_score(y_bin_te, gnb_bin_pred))
print("F1 (stress class):", f1_score(y_bin_te, gnb_bin_pred))
print(classification_report(y_bin_te, gnb_bin_pred, target_names=["non-stress", "stress"]))

Dummy (most-frequent) binary accuracy: 0.52

Gaussian NB — binary
accuracy: 0.472
F1 (stress class): 0.2978723404255319
              precision    recall  f1-score   support

  non-stress       0.47      0.75      0.58       300
      stress       0.48      0.22      0.30       325

    accuracy                           0.47       625
   macro avg       0.48      0.48      0.44       625
weighted avg       0.48      0.47      0.43       625



In [16]:
gnb_multi = GaussianNB()
gnb_multi.fit(X_train, y_multi_tr)
gnb_multi_pred = gnb_multi.predict(X_test)
print("Gaussian NB — 4-class")
print("accuracy:", accuracy_score(y_multi_te, gnb_multi_pred))
print("F1-macro:", f1_score(y_multi_te, gnb_multi_pred, average="macro"))
print(classification_report(y_multi_te, gnb_multi_pred, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cm_bin = confusion_matrix(y_bin_te, gnb_bin_pred)
sns.heatmap(cm_bin, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["non-stress", "stress"], yticklabels=["non-stress", "stress"])
axes[0].set_title("GNB binary — confusion (held-out subjects)")
axes[0].set_xlabel("predicted"); axes[0].set_ylabel("true")
labels_order = ["rest", "no stress", "time pressure", "interruption"]
cm_m = confusion_matrix(y_multi_te, gnb_multi_pred, labels=labels_order)
sns.heatmap(cm_m, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            xticklabels=labels_order, yticklabels=labels_order)
axes[1].set_title("GNB 4-class — confusion (held-out subjects)")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(FIG_DIR / "07_gnb_confusion.png", dpi=150)
plt.show()

Gaussian NB — 4-class
accuracy: 0.3632
F1-macro: 0.28378848346597246
               precision    recall  f1-score   support

 interruption       0.36      0.25      0.29       196
    no stress       0.34      0.73      0.47       209
         rest       0.58      0.27      0.37        91
time pressure       0.00      0.00      0.00       129

     accuracy                           0.36       625
    macro avg       0.32      0.31      0.28       625
 weighted avg       0.31      0.36      0.30       625



/var/folders/ny/s5wjrp7d0l93sy7y1qnr6sz80000gn/T/ipykernel_9349/3995693987.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
gkf = GroupKFold(n_splits=5)
cv_rows = []
for fold, (tr, te) in enumerate(gkf.split(X, y_bin, groups), start=1):
    sc = StandardScaler()
    Xt, Xe = sc.fit_transform(X[tr]), sc.transform(X[te])
    pred = GaussianNB().fit(Xt, y_bin[tr]).predict(Xe)
    cv_rows.append({
        "fold": fold,
        "accuracy": accuracy_score(y_bin[te], pred),
        "f1": f1_score(y_bin[te], pred),
        "n_test_subjects": len(set(groups[te])),
    })
cv_df = pd.DataFrame(cv_rows)
print(cv_df)
print("\nGroupKFold GNB binary — mean accuracy", cv_df.accuracy.mean().round(3),
      "±", cv_df.accuracy.std().round(3),
      "  mean F1", cv_df.f1.mean().round(3), "±", cv_df.f1.std().round(3))

   fold  accuracy        f1  n_test_subjects
0     1  0.609795  0.508946                5
1     2  0.528571  0.530806                5
2     3  0.538827  0.477558                5
3     4  0.492038  0.360721                5
4     5  0.553398  0.572755                5

GroupKFold GNB binary — mean accuracy 0.545 ± 0.043   mean F1 0.49 ± 0.08


### Unsupervised generative structure (k-means + GMM)
Fit **without using labels**, then ask whether the recovered 4 clusters match the 4 experimental
conditions (Adjusted Rand Index / NMI vs the true `label`). Near-zero ARI means the
physiological feature cloud is **not** organised into four well-separated Gaussians matching
the experimental design — consistent with the overlapping boxplots in Step 3.

In [18]:
X_all = StandardScaler().fit_transform(X)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_all)
gmm = GaussianMixture(n_components=4, covariance_type="full", random_state=42).fit(X_all)
gmm_labels = gmm.predict(X_all)

print("KMeans vs 4-class labels  ARI:", round(adjusted_rand_score(y_multi, kmeans.labels_), 4),
      " NMI:", round(normalized_mutual_info_score(y_multi, kmeans.labels_), 4))
print("GMM    vs 4-class labels  ARI:", round(adjusted_rand_score(y_multi, gmm_labels), 4),
      " NMI:", round(normalized_mutual_info_score(y_multi, gmm_labels), 4))
print("KMeans vs binary          ARI:", round(adjusted_rand_score(y_bin, kmeans.labels_), 4))
print("GMM    vs binary          ARI:", round(adjusted_rand_score(y_bin, gmm_labels), 4))

KMeans vs 4-class labels  ARI: 0.0223  NMI: 0.0259
GMM    vs 4-class labels  ARI: 0.0033  NMI: 0.0045
KMeans vs binary          ARI: 0.0019
GMM    vs binary          ARI: 0.0004


### Class-conditional GMM classifier
Instead of one Gaussian per class (Naive Bayes), fit a 2-component GMM on each binary class
and classify by maximum log-likelihood + class prior. This is still a generative model, just
with a more flexible density.

In [19]:
gmm0 = GaussianMixture(n_components=2, random_state=42).fit(X_train[y_bin_tr == 0])
gmm1 = GaussianMixture(n_components=2, random_state=42).fit(X_train[y_bin_tr == 1])
prior0 = np.mean(y_bin_tr == 0)
prior1 = 1.0 - prior0
ll0 = gmm0.score_samples(X_test) + np.log(prior0)
ll1 = gmm1.score_samples(X_test) + np.log(prior1)
gmm_clf_pred = (ll1 > ll0).astype(int)
print("Class-conditional GMM (binary) accuracy:", accuracy_score(y_bin_te, gmm_clf_pred))
print("F1 (stress class):", f1_score(y_bin_te, gmm_clf_pred))
print(classification_report(y_bin_te, gmm_clf_pred, target_names=["non-stress", "stress"]))

Class-conditional GMM (binary) accuracy: 0.5232
F1 (stress class): 0.565597667638484
              precision    recall  f1-score   support

  non-stress       0.50      0.44      0.47       300
      stress       0.54      0.60      0.57       325

    accuracy                           0.52       625
   macro avg       0.52      0.52      0.52       625
weighted avg       0.52      0.52      0.52       625



**Takeaway for the report.** On held-out *people*, Gaussian Naive Bayes does **not** beat a
most-frequent dummy classifier (and 5-fold GroupKFold sits around ~54% — barely above chance
on a ~50/50 binary task). Clustering ARI is ~0. Unsupervised Gaussians in this feature space
do not recover experimental conditions. That is expected from Step 3 (overlapping class
distributions) and is itself a valid result: these minute-level autonomic features are a weak
linear/Gaussian signal. Discriminative models in Step 5 can still try non-linear boundaries,
but we should not expect large gains without richer windowed HRV features or subject-specific
baselines beyond SCL_z.

## 5. Discriminative modeling

Discriminative models learn **p(y | x)** / a decision boundary directly, without modelling
the class-conditional density. Same subject-aware split and scaled features as Step 4.

- **Logistic regression** — linear boundary (probabilistic)
- **Perceptron** — linear boundary (mistake-driven)
- **Decision tree** (max depth 5) — axis-aligned non-linear rules
- **Random forest** — ensemble of trees (syllabus: ensemble methods)

`class_weight="balanced"` is used because rest is the smallest 4-class category.

In [20]:
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

disc_models = {
    "Dummy (most frequent)": DummyClassifier(strategy="most_frequent", random_state=42),
    "Logistic regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision tree (depth=5)": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42),
    "Random forest": RandomForestClassifier(
        n_estimators=200, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "Gaussian NB": GaussianNB(),
}

heldout_rows = []
fitted = {}
for name, model in disc_models.items():
    model.fit(X_train, y_bin_tr)
    pred = model.predict(X_test)
    fitted[name] = pred
    heldout_rows.append({
        "model": name,
        "heldout_acc": accuracy_score(y_bin_te, pred),
        "heldout_f1": f1_score(y_bin_te, pred, zero_division=0),
        "heldout_f1_macro": f1_score(y_bin_te, pred, average="macro", zero_division=0),
    })
heldout_df = pd.DataFrame(heldout_rows)
print("Binary classification — held-out subjects")
heldout_df

Binary classification — held-out subjects


,model,heldout_acc,heldout_f1,heldout_f1_macro
0,Dummy (most frequent),0.5200,0.684211,0.342105
1,Logistic regression,0.4368,0.391003,0.433597
2,Perceptron,0.4224,0.465185,0.418680
3,Decision tree (depth=5),0.5664,0.504570,0.559540
4,Random forest,0.4928,0.527571,0.490038
5,Gaussian NB,0.4720,0.297872,0.437398


In [21]:
print("Decision tree (binary) classification report")
print(classification_report(y_bin_te, fitted["Decision tree (depth=5)"],
                            target_names=["non-stress", "stress"], zero_division=0))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name in zip(axes, ["Logistic regression", "Decision tree (depth=5)", "Random forest"]):
    cm = confusion_matrix(y_bin_te, fitted[name])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", ax=ax,
                xticklabels=["non-stress", "stress"], yticklabels=["non-stress", "stress"])
    ax.set_title(name)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout()
plt.savefig(FIG_DIR / "08_discriminative_confusion.png", dpi=150)
plt.show()

Decision tree (binary) classification report
              precision    recall  f1-score   support

  non-stress       0.54      0.72      0.61       300
      stress       0.62      0.42      0.50       325

    accuracy                           0.57       625
   macro avg       0.58      0.57      0.56       625
weighted avg       0.58      0.57      0.56       625



/var/folders/ny/s5wjrp7d0l93sy7y1qnr6sz80000gn/T/ipykernel_9349/366179.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
tree = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)
tree.fit(X_train, y_bin_tr)
rf = disc_models["Random forest"]
imp = pd.DataFrame({
    "feature": feature_cols,
    "tree_importance": tree.feature_importances_,
    "rf_importance": rf.feature_importances_,
}).sort_values("rf_importance", ascending=False)
print(imp)

fig, ax = plt.subplots(figsize=(6, 3.5))
imp.set_index("feature")[["tree_importance", "rf_importance"]].plot(kind="barh", ax=ax)
ax.set_title("Feature importance (binary stress task)")
plt.tight_layout()
plt.savefig(FIG_DIR / "09_feature_importance.png", dpi=150)
plt.show()

          feature  tree_importance  rf_importance
2           SCL_z         0.460255       0.541190
1           RMSSD         0.269363       0.173047
3  RMSSD_HR_ratio         0.097270       0.156596
0              HR         0.173112       0.129167


/var/folders/ny/s5wjrp7d0l93sy7y1qnr6sz80000gn/T/ipykernel_9349/1798489477.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
# 4-class held-out (same people)
multi_rows = []
for name, model in disc_models.items():
    model.fit(X_train, y_multi_tr)
    pred = model.predict(X_test)
    multi_rows.append({
        "model": name,
        "heldout_acc": accuracy_score(y_multi_te, pred),
        "heldout_f1_macro": f1_score(y_multi_te, pred, average="macro", zero_division=0),
    })
multi_df = pd.DataFrame(multi_rows)
print("4-class classification — held-out subjects")
multi_df

4-class classification — held-out subjects


,model,heldout_acc,heldout_f1_macro
0,Dummy (most frequent),0.3344,0.125300
1,Logistic regression,0.2528,0.223605
2,Perceptron,0.2192,0.168655
3,Decision tree (depth=5),0.3072,0.236446
4,Random forest,0.2048,0.205659
5,Gaussian NB,0.3632,0.283788


## 6. Evaluation & comparison

Subject-grouped 5-fold CV is the number to trust more than any single held-out split
(the 5 test people in Step 4 happened to include p8, who has no real HR/RMSSD).

In [24]:
gkf = GroupKFold(n_splits=5)
cv_summary = []
ctors = {
    "Dummy (most frequent)": lambda: DummyClassifier(strategy="most_frequent", random_state=42),
    "Logistic regression": lambda: LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Perceptron": lambda: Perceptron(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision tree (depth=5)": lambda: DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42),
    "Random forest": lambda: RandomForestClassifier(
        n_estimators=200, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "Gaussian NB": lambda: GaussianNB(),
}
for name, ctor in ctors.items():
    accs, f1s = [], []
    for tr, te in gkf.split(X, y_bin, groups):
        sc = StandardScaler()
        Xt, Xe = sc.fit_transform(X[tr]), sc.transform(X[te])
        pred = ctor().fit(Xt, y_bin[tr]).predict(Xe)
        accs.append(accuracy_score(y_bin[te], pred))
        f1s.append(f1_score(y_bin[te], pred, zero_division=0))
    cv_summary.append({
        "model": name,
        "cv_acc_mean": np.mean(accs),
        "cv_acc_std": np.std(accs),
        "cv_f1_mean": np.mean(f1s),
        "cv_f1_std": np.std(f1s),
    })
cv_summary_df = pd.DataFrame(cv_summary).sort_values("cv_acc_mean", ascending=False)
comparison = cv_summary_df.merge(heldout_df, on="model", how="left")
comparison.to_csv(PROCESSED_DIR / "model_comparison_binary.csv", index=False)
print(comparison.round(3).to_string(index=False))

                  model  cv_acc_mean  cv_acc_std  cv_f1_mean  cv_f1_std  heldout_acc  heldout_f1  heldout_f1_macro
          Random forest        0.566       0.037       0.595      0.061        0.493       0.528             0.490
            Gaussian NB        0.545       0.038       0.490      0.072        0.472       0.298             0.437
Decision tree (depth=5)        0.537       0.050       0.584      0.048        0.566       0.505             0.560
  Dummy (most frequent)        0.529       0.008       0.692      0.006        0.520       0.684             0.342
    Logistic regression        0.506       0.036       0.459      0.053        0.437       0.391             0.434
             Perceptron        0.497       0.026       0.478      0.105        0.422       0.465             0.419


In [25]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = cv_summary_df.sort_values("cv_acc_mean")
ax.barh(plot_df["model"], plot_df["cv_acc_mean"], xerr=plot_df["cv_acc_std"], color="#4C72B0", alpha=0.85)
ax.axvline(0.5, color="grey", ls="--", lw=1, label="chance (0.50)")
ax.set_xlabel("GroupKFold accuracy (binary is_stress)")
ax.set_title("Model comparison — subject-grouped 5-fold CV")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "10_model_comparison.png", dpi=150)
plt.show()

/var/folders/ny/s5wjrp7d0l93sy7y1qnr6sz80000gn/T/ipykernel_9349/1412685211.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Takeaway.** Random forest is the strongest of the set (~57% GroupKFold accuracy) but the
lift over a most-frequent dummy (~53%) is small. Linear discriminative models (logistic
regression, perceptron) do not beat chance on held-out people — same story as Gaussian NB.
Trees help a little because they can use subject-scale quirks (especially `SCL_z`), not
because stress is linearly separable in HR/RMSSD/SCL.

This is a defensible coursework result: the syllabus models were applied correctly with a
leakage-safe protocol, and the data's limited linear signal is quantified rather than hidden.
Richer windowed HRV features or a subject-adaptive model would be the natural next experiment
(out of scope here, and complementary to the image/CNN work on Blood Cell detection).

## 5b. Can we make it more accurate?

The weak results above are mostly a **feature** problem, not an algorithm problem.
HR 70 bpm means different things for different people. Stress in this literature is usually
detected as a **deviation from that person's own baseline**, which we already did for SCL
(`SCL_z`) but not for HR/RMSSD.

Two leakage-safe upgrades (computed **within subject**, never using test labels):

1. **Within-subject z-score** of HR and RMSSD (`HR_z`, `RMSSD_z`) — same transform as `SCL_z`.
2. **Rest-relative deltas** — each minute minus that subject's median *rest* HR/RMSSD/SCL.

A third check: train only on rows with **real** (non-imputed) HR/RMSSD. Missingness is ~50%
in every class, so this is not secretly dropping one condition.

In [26]:
feat_imp = features_minute.copy()
feat_imp["HR_z"] = feat_imp.groupby("subject")["HR"].transform(
    lambda s: (s - s.mean()) / (s.std(ddof=0) + 1e-9)
)
feat_imp["RMSSD_z"] = feat_imp.groupby("subject")["RMSSD"].transform(
    lambda s: (s - s.mean()) / (s.std(ddof=0) + 1e-9)
)

rest_med = (
    feat_imp.loc[feat_imp["label"] == "rest"]
    .groupby("subject")[["HR", "RMSSD", "SCL"]]
    .median()
    .rename(columns={"HR": "HR_rest", "RMSSD": "RMSSD_rest", "SCL": "SCL_rest"})
)
feat_imp = feat_imp.merge(rest_med, on="subject", how="left")
feat_imp["HR_minus_rest"] = feat_imp["HR"] - feat_imp["HR_rest"]
feat_imp["RMSSD_minus_rest"] = feat_imp["RMSSD"] - feat_imp["RMSSD_rest"]
feat_imp["SCL_minus_rest"] = feat_imp["SCL"] - feat_imp["SCL_rest"]
feat_imp.to_csv(PROCESSED_DIR / "features_engineered.csv", index=False)
print(feat_imp[["HR_z", "RMSSD_z", "HR_minus_rest"]].describe().round(3))

           HR_z   RMSSD_z  HR_minus_rest
count  3140.000  3140.000       3140.000
mean      0.000    -0.000          0.811
std       0.961     0.961          3.704
min      -7.360    -3.880        -22.000
25%      -0.118    -0.241          0.000
50%       0.000    -0.139          0.000
75%       0.106     0.000          2.000
max      10.432     9.588         25.500


In [27]:
def grouped_cv(df, cols, y_col="is_stress"):
    X_ = df[cols].to_numpy()
    y_ = df[y_col].to_numpy()
    g_ = df["subject"].to_numpy()
    rows = []
    ctors = {
        "Dummy": lambda: DummyClassifier(strategy="most_frequent", random_state=42),
        "Logistic regression": lambda: LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
        "Gaussian NB": lambda: GaussianNB(),
        "Random forest": lambda: RandomForestClassifier(
            n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1
        ),
    }
    gkf_ = GroupKFold(n_splits=5)
    for name, ctor in ctors.items():
        accs, f1s = [], []
        for tr, te in gkf_.split(X_, y_, g_):
            sc = StandardScaler()
            Xt, Xe = sc.fit_transform(X_[tr]), sc.transform(X_[te])
            pred = ctor().fit(Xt, y_[tr]).predict(Xe)
            accs.append(accuracy_score(y_[te], pred))
            f1s.append(f1_score(y_[te], pred, zero_division=0))
        rows.append({
            "model": name,
            "cv_acc_mean": np.mean(accs),
            "cv_acc_std": np.std(accs),
            "cv_f1_mean": np.mean(f1s),
            "cv_f1_std": np.std(f1s),
        })
    return pd.DataFrame(rows)

orig_cols = ["HR", "RMSSD", "SCL_z", "RMSSD_HR_ratio"]
z_cols = ["HR_z", "RMSSD_z", "SCL_z", "RMSSD_HR_ratio"]
delta_cols = ["HR_minus_rest", "RMSSD_minus_rest", "SCL_minus_rest"]

a = grouped_cv(feat_imp, orig_cols).assign(setup="A original (absolute HR/RMSSD)")
b = grouped_cv(feat_imp, z_cols).assign(setup="B within-subject z-scores")
c = grouped_cv(feat_imp, delta_cols).assign(setup="C rest-relative deltas")
cc = feat_imp[feat_imp["HR_imputed_flag"] == 0].reset_index(drop=True)
g = grouped_cv(cc, z_cols).assign(setup="G complete-case + z-scores")

improve_df = pd.concat([a, b, c, g], ignore_index=True)
improve_df.to_csv(PROCESSED_DIR / "model_improvement_binary.csv", index=False)
print(improve_df.round(3).to_string(index=False))

              model  cv_acc_mean  cv_acc_std  cv_f1_mean  cv_f1_std                          setup
              Dummy        0.529       0.008       0.692      0.006 A original (absolute HR/RMSSD)
Logistic regression        0.506       0.036       0.459      0.053 A original (absolute HR/RMSSD)
        Gaussian NB        0.545       0.038       0.490      0.072 A original (absolute HR/RMSSD)
      Random forest        0.562       0.052       0.572      0.074 A original (absolute HR/RMSSD)
              Dummy        0.529       0.008       0.692      0.006      B within-subject z-scores
Logistic regression        0.555       0.033       0.541      0.038      B within-subject z-scores
        Gaussian NB        0.601       0.024       0.672      0.010      B within-subject z-scores
      Random forest        0.604       0.037       0.640      0.030      B within-subject z-scores
              Dummy        0.529       0.008       0.692      0.006         C rest-relative deltas
Logistic r

In [28]:
rf_rows = improve_df[improve_df.model == "Random forest"]
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(rf_rows["setup"], rf_rows["cv_acc_mean"], xerr=rf_rows["cv_acc_std"], color="#55A868")
ax.axvline(0.5, color="grey", ls="--", lw=1)
ax.set_xlabel("Random forest GroupKFold accuracy")
ax.set_title("Accuracy lift from within-subject features")
plt.tight_layout()
plt.savefig(FIG_DIR / "11_accuracy_improvements.png", dpi=150)
plt.show()

/var/folders/ny/s5wjrp7d0l93sy7y1qnr6sz80000gn/T/ipykernel_9349/14864450.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**What actually moved the needle**

- Within-subject `HR_z` / `RMSSD_z` (setup B): Random forest **~60%** vs **~56%** on absolute
  HR/RMSSD, and Gaussian NB jumps to match it. Logistic regression now **beats** the dummy
  baseline. Personalizing the features helps more than swapping algorithms.
- Rest-relative deltas alone are weaker than z-scores.
- Complete-case (only minutes with a real ECG-derived HR, n≈1,490): RF **~70%**. Fair, because
  HR is missing at similar rates in every class — but the sample is smaller, so report both
  numbers and do not claim 70% on the full 3,140 rows.

We still will not reach CNN-like accuracy: these are four overlapping physiological streams
from 25 people. The honest story for the report is: **absolute HR is a poor stress detector;
deviation from the person's own baseline is better, and trees pick that up.**